# Export the shipped Greek neural lemmatizer (`grc-lemma-neural`)

Trains the gold-only GreTa seq2seq (the 76.3%-unseen recipe) on **all** gold, exports ONNX, and packages the bundle `pyaegean`'s `[neural]` backend fetches. **Run all** on a GPU runtime, then upload the downloaded `grc-lemma-neural.tar.gz` to a GitHub release and report its sha256.

In [ ]:
!nvidia-smi -L
%pip -q install 'transformers>=4.46' 'datasets>=2.19' 'optimum[onnxruntime]>=1.20' accelerate sentencepiece protobuf onnx onnxruntime

## 0 · GPU + precision

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU, reconnect.'
USE_BF16 = torch.cuda.is_bf16_supported()
BS = 128 if USE_BF16 else 16
print(f'torch {torch.__version__} | GPU {torch.cuda.get_device_name(0)} | bf16={USE_BF16}')

## 1 · Upload `prod_data.zip` (prod_train.jsonl + lookup.json.gz)

In [ ]:
import json, zipfile, pathlib
from google.colab import files
up = files.upload()  # pick prod_data.zip
zipfile.ZipFile(next(n for n in up if n.endswith('.zip'))).extractall('.')
DATA = pathlib.Path('data') if pathlib.Path('data/prod_train.jsonl').exists() else pathlib.Path('.')
train_rows = [json.loads(l) for l in open(DATA / 'prod_train.jsonl', encoding='utf-8')]
print('form->lemma pairs:', len(train_rows))

## 2 · Tokenizer + model (`bowphs/GreTa`), with pad/eos registered

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MODEL = 'bowphs/GreTa'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
PAD_ID, EOS_ID = 0, 1  # GreTa ships a bare tokenizer; config.json defines pad=0, eos=1
tokenizer.pad_token = tokenizer.convert_ids_to_tokens(PAD_ID)
tokenizer.eos_token = tokenizer.convert_ids_to_tokens(EOS_ID)
for cfg in (model.config, model.generation_config):
    cfg.pad_token_id = PAD_ID; cfg.eos_token_id = EOS_ID; cfg.decoder_start_token_id = PAD_ID
assert tokenizer.pad_token_id == PAD_ID and tokenizer.eos_token_id == EOS_ID

## 3 · Tokenize (force eos on targets)

In [ ]:
from datasets import Dataset
ML = 32
APPEND_EOS = tokenizer(text_target='abc')['input_ids'][-1] != EOS_ID
def prep(b):
    enc = tokenizer(b['form'], max_length=ML, truncation=True)
    lab = tokenizer(text_target=b['lemma'], max_length=ML - APPEND_EOS, truncation=True)['input_ids']
    enc['labels'] = [x + [EOS_ID] for x in lab] if APPEND_EOS else lab
    return enc
ds = Dataset.from_list(train_rows).map(prep, batched=True, remove_columns=['form', 'lemma'])
ds = ds.train_test_split(test_size=2000, seed=0)  # small slice for monitoring only
print('APPEND_EOS', APPEND_EOS, '| train', len(ds['train']))

## 4 · Fine-tune (single-phase, the 76.3% recipe)
T5 uses **bf16, not fp16**; ~10 epochs on all gold, a few minutes on an H100.

In [ ]:
import numpy as np
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq)
collator = DataCollatorForSeq2Seq(tokenizer, model=model)
def compute_metrics(ep):
    preds, labels = ep
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    dp = tokenizer.batch_decode(preds, skip_special_tokens=True)
    dl = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {'exact': float(np.mean([a.strip() == b.strip() for a, b in zip(dp, dl)]))}
args = Seq2SeqTrainingArguments(
    output_dir='out', learning_rate=3e-4, lr_scheduler_type='cosine',
    per_device_train_batch_size=BS, per_device_eval_batch_size=BS*2,
    num_train_epochs=10, weight_decay=0.01, warmup_ratio=0.06,
    bf16=USE_BF16, fp16=False, tf32=USE_BF16, optim='adamw_torch_fused',
    dataloader_num_workers=2, predict_with_generate=True, generation_max_length=ML,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='exact', greater_is_better=True,
    logging_steps=200, report_to='none')
Seq2SeqTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['test'],
               data_collator=collator, processing_class=tokenizer,
               compute_metrics=compute_metrics).train()
model.save_pretrained('out_model'); tokenizer.save_pretrained('out_model')

## 5 · Export ONNX (encoder + decoder)

In [ ]:
!optimum-cli export onnx --model out_model --task text2text-generation onnx
import os
print(sorted(os.listdir('onnx')))
assert os.path.exists('onnx/decoder_model.onnx'), \
    'no decoder_model.onnx (the no-past decoder the torch-free loop needs) — retry export with --no-post-process'

## 6 · Assemble + download `grc-lemma-neural.tar.gz`
The four files the package loads, tarred at the archive root so it unpacks straight into the cache. **Report the printed sha256** — I pin it in the data layer.

In [ ]:
import os, shutil, tarfile, hashlib
B = 'grc-lemma-neural'
os.makedirs(B, exist_ok=True)
for fn in ['encoder_model.onnx', 'decoder_model.onnx', 'tokenizer.json']:
    shutil.copy(os.path.join('onnx', fn), os.path.join(B, fn))
shutil.copy(str(DATA / 'lookup.json.gz'), os.path.join(B, 'lookup.json.gz'))
with tarfile.open(B + '.tar.gz', 'w:gz') as tf:
    for fn in sorted(os.listdir(B)):
        tf.add(os.path.join(B, fn), arcname=fn)  # arcname=fn -> files at archive root
sz = os.path.getsize(B + '.tar.gz') // (1024 * 1024)
sha = hashlib.sha256(open(B + '.tar.gz', 'rb').read()).hexdigest()
print(f'{B}.tar.gz  {sz} MB\nsha256={sha}')
files.download(B + '.tar.gz')

## Next
1. Upload `grc-lemma-neural.tar.gz` to a GitHub release on `ryanpavlicek/pyaegean`.
2. Report the **sha256** and the release asset URL — I pin them in `aegean.data._REMOTE`.
3. Then `pip install 'pyaegean[neural]'` + `greek.use_neural_lemmatizer()` fetches and runs it.